In [ ]:
from lsst.summit.utils import ConsDbClient

In [ ]:
import numpy as np
from astropy.table import Table, join, vstack
from astropy.time import Time
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, get_sun, get_body
import astropy.units as u

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
%matplotlib widget

import pandas as pd

from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma

from tqdm.notebook import tqdm

import os
import pandas as pd
from matplotlib.cm import get_cmap

In [ ]:
os.environ["no_proxy"] += ",.consdb"

In [ ]:
url="http://consdb-pq.consdb:8080/consdb"

In [ ]:
consdb=ConsDbClient(url)

In [ ]:
# Query both consDB tables
exposure = consdb.query("SELECT * FROM cdb_lsstcam.exposure WHERE science_program IN ('BLOCK-407','BLOCK-408','BLOCK-417','BLOCK-417','BLOCK-419') AND day_obs > 20260101")

In [ ]:
exposure.columns

In [ ]:
def rubin_location():
    # Rubin Observatory / Cerro Pachón (approx)
    return EarthLocation.from_geodetic(
        lon=-70.7494 * u.deg,
        lat=-30.2446 * u.deg,
        height=2663 * u.m,
    )

def _as_time(x):
    """Best-effort conversion of a scalar table value to astropy.time.Time (UTC)."""
    if isinstance(x, Time):
        return x
    # common cases: ISO string, numpy datetime64, python datetime
    return Time(x, scale="utc")

def moon_illumination_fraction(t):
    """
    Illuminated fraction of the Moon's disk (0..1), computed from
    geocentric Sun–Moon elongation.
    """
    t = t if isinstance(t, Time) else Time(t, scale="utc")
    moon = get_body("moon", t)    # geocentric apparent
    sun  = get_sun(t)             # geocentric apparent
    elong = moon.separation(sun)
    alpha = elong.to_value(u.rad)
    return float(0.5 * (1.0 - np.cos(alpha)))  # New~0, Full~1

In [ ]:
def first_exposure_per_day(tab: Table, time_col: str = "obs_start") -> Table:
    """
    Return a table with one row per day_obs: the earliest exposure by time_col.
    """
    if "day_obs" not in tab.colnames:
        raise KeyError("Table must contain a 'day_obs' column.")
    if time_col not in tab.colnames:
        raise KeyError(f"Table must contain '{time_col}' column.")

    out_rows = []
    for day in np.unique(tab["day_obs"]):
        sub = tab[tab["day_obs"] == day]

        # Convert times to astropy Time for robust sorting
        times = Time([_as_time(v).utc.isot for v in sub[time_col]], scale="utc")
        i0 = int(np.argmin(times.mjd))
        out_rows.append(sub[i0])

    return Table(rows=out_rows, names=tab.colnames, meta=getattr(tab, "meta", None))

In [ ]:
def add_sun_moon_metrics(
    tab_first: Table,
    *,
    time_col: str = "obs_start",
    tel_az_col: str = "azimuth",
    tel_alt_col: str = "altitude",
    location: EarthLocation | None = None,
    with_refraction: bool = False,
    pressure_hpa: float = 750.0,
    temperature_C: float = 5.0,
    relative_humidity: float = 0.2,
) -> Table:
    """
    For each row, compute Sun alt/az, Moon alt/az, Moon illumination, and
    separation between telescope pointing (Az/Alt) and the Moon.
    """
    location = location or rubin_location()

    # Build AltAz frame options
    def make_altaz_frame(t):
        if with_refraction:
            return AltAz(
                obstime=t,
                location=location,
                pressure=pressure_hpa * u.hPa,
                temperature=temperature_C * u.deg_C,
                relative_humidity=relative_humidity,
                obswl=0.55 * u.micron,
            )
        else:
            return AltAz(obstime=t, location=location)

    # Prepare output columns
    sun_alt = []
    sun_az = []
    moon_alt = []
    moon_az = []
    moon_sep = []
    illum_frac = []
    illum_pct = []
    tel_az_out = []
    tel_alt_out = []

    for row in tab_first:
        t = _as_time(row[time_col])
        altaz_frame = make_altaz_frame(t)

        # Sun (geocentric -> topocentric via transform)
        sun_altaz = get_sun(t).transform_to(altaz_frame)

        # Moon (topocentric if you pass location)
        moon_icrs = get_body("moon", t, location=location)
        moon_altaz = moon_icrs.transform_to(altaz_frame)

        # Telescope pointing in local horizon coordinates
        tel_az = float(row[tel_az_col])
        tel_alt = float(row[tel_alt_col])
        pointing = SkyCoord(az=tel_az * u.deg, alt=tel_alt * u.deg, frame=altaz_frame)

        # Separation in the AltAz frame (great-circle)
        sep = pointing.separation(moon_altaz)

        # Illumination (geocentric, standard definition)
        f = moon_illumination_fraction(t)

        sun_alt.append(float(sun_altaz.alt.to_value(u.deg)))
        sun_az.append(float(sun_altaz.az.to_value(u.deg)))
        moon_alt.append(float(moon_altaz.alt.to_value(u.deg)))
        moon_az.append(float(moon_altaz.az.to_value(u.deg)))
        moon_sep.append(float(sep.to_value(u.deg)))
        illum_frac.append(float(f))
        illum_pct.append(float(100.0 * f))
        tel_az_out.append(tel_az)
        tel_alt_out.append(tel_alt)

    out = tab_first.copy()
    out["tel_az_deg"] = tel_az_out
    out["tel_alt_deg"] = tel_alt_out
    out["sun_alt_deg"] = sun_alt
    out["sun_az_deg"] = sun_az
    out["moon_alt_deg"] = moon_alt
    out["moon_az_deg"] = moon_az
    out["moon_sep_deg"] = moon_sep
    out["moon_illum_frac"] = illum_frac
    out["moon_illum_percent"] = illum_pct

    return out

In [ ]:
first = first_exposure_per_day(exposure, time_col="obs_start")

summary = add_sun_moon_metrics(
    first,
    time_col="obs_start",
    tel_az_col="azimuth",
    tel_alt_col="altitude",
    with_refraction=False,
)

summary = summary[summary['sun_alt_deg'] > -30]


# Show a compact view
summary[
    "day_obs", "exposure_id", "obs_start",
    "tel_az_deg", "tel_alt_deg",
    "sun_az_deg", "sun_alt_deg",
    "moon_az_deg", "moon_alt_deg",
    "moon_sep_deg", "moon_illum_percent",
]

In [ ]:
summary.sort("sun_alt_deg", reverse=True)

In [ ]:
summary[
    "day_obs", "exposure_id", "obs_start",
    "tel_az_deg", "tel_alt_deg",
    "sun_az_deg", "sun_alt_deg",
    "moon_az_deg", "moon_alt_deg",
    "moon_sep_deg", "moon_illum_percent",
]

In [ ]:
import pandas as pd

df = summary.copy()

df["day_obs_dt"] = pd.to_datetime(
    df["day_obs"].astype(str),
    format="%Y%m%d"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1) Convert table -> pandas DataFrame
df = summary.to_pandas()  # Astropy Table supports this

# 2) Parse day_obs (YYYYMMDD int/string) -> datetime
df["day_obs_dt"] = pd.to_datetime(df["day_obs"].astype(str), format="%Y%m%d")

# 3) Aggregate per night (median is a good default), then EWMA
nightly = (
    df.groupby("day_obs_dt")["sun_alt_deg"]
      .median()
      .sort_index()
)

# Option A: EWMA by "nights" (works fine if you have most nights)
ewma = nightly.ewm(span=5, adjust=False).mean()

# Option B (often better): time-aware EWMA, handles missing nights naturally
# ewma = nightly.ewm(halflife="7D", times=nightly.index).mean()

# 4) Plot
plt.figure(figsize=(10, 4))

plt.plot(
    ewma.index,
    ewma.values,
    linewidth=2.0,
    alpha=0.6,
    zorder=1,
    label="EWMA (span=10 nights)",
)

plt.scatter(
    nightly.index,
    nightly.values,
    s=25,
    alpha=0.7,
    zorder=3,
    edgecolors="0.2",
    label="Nightly median",
)

plt.axhline(-12, color="g", linestyle="--", linewidth=1, alpha=0.6)

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xlabel("Date)")
plt.ylabel("Solar Elevation (deg)")
plt.title("Starting Solar Elevation of Scheduler-driven Observations")
plt.grid(True, alpha=0.3)
plt.ylim(-31,-8.5)
ax.set_yticks([-30, -27, -24, -21, -18, -15, -12, -9])
ax.set_yticklabels(["-30", "-27", "-24", "-21", "-18", "-15", "-12", "-9"])
#plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from astropy.time import Time
from astropy.coordinates import AltAz, get_sun
import astropy.units as u


from rubin_nights.dayobs_utils import day_obs_sunset_sunrise

def evening_sun_alt_crossing(day_obs, target_alt=-12.0):
    """Return evening UTC time when Sun crosses target_alt for a day_obs."""
    sunset, sunrise = day_obs_sunset_sunrise(
        day_obs,
        sun_alt=target_alt,
    )
    return sunset

# 1) Convert table -> pandas DataFrame
df = summary.to_pandas()

# 2) Parse day_obs
df["day_obs_dt"] = pd.to_datetime(
    df["day_obs"].astype(str),
    format="%Y%m%d",
)

# 3) Compute exact evening -12 deg time per row/night
df["sun_minus12_time"] = [
    evening_sun_alt_crossing(day_obs, target_alt=-12.0)
    for day_obs in df["day_obs"]
]

# 4) Minutes from -12 deg crossing to survey start
df["minutes_after_minus12"] = [
    (Time(obs_start, scale="utc") - t12).to_value(u.min)
    if not isinstance(t12, float) or not np.isnan(t12)
    else np.nan
    for obs_start, t12 in zip(df["obs_start"], df["sun_minus12_time"])
]

# Keep only nights that started within 60 minutes after -12 deg
df = df[df["minutes_after_minus12"] <= 60].copy()

# 5) One value per night
nightly = (
    df.groupby("day_obs_dt")["minutes_after_minus12"]
      .median()
      .sort_index()
)

# 6) EWMA
ewma = nightly.ewm(span=5, adjust=False).mean()

# 7) Plot
plt.figure(figsize=(10, 4))

plt.plot(
    ewma.index,
    ewma.values,
    linewidth=2.0,
    alpha=0.6,
    zorder=1,
    label="EWMA (span=5 nights)",
)

plt.scatter(
    nightly.index,
    nightly.values,
    s=25,
    alpha=0.7,
    zorder=3,
    edgecolors="0.2",
    label="Nightly median",
)

plt.axhline(0, color="g", linestyle="--", linewidth=1, alpha=0.6)

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xlabel("Date")
plt.ylabel("Minutes after Sun elevation = -12°")
plt.title("Survey Start Delay Relative to Evening -12° Solar Elevation")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

***
Initial Alignment
***

In [ ]:
# Query both consDB tables
alignment_exposures = consdb.query("SELECT * FROM cdb_lsstcam.exposure WHERE science_program = 'BLOCK-T539' AND day_obs > 20251001")

In [ ]:
alignment_exposures.columns

In [ ]:
first = first_exposure_per_day(alignment_exposures, time_col="obs_start")

alignment_summary = add_sun_moon_metrics(
    first,
    time_col="obs_start",
    tel_az_col="azimuth",
    tel_alt_col="altitude",
    with_refraction=False,
)

alignment_summary = alignment_summary[alignment_summary['sun_alt_deg'] > -12]

# Show a compact view
alignment_summary[
    "day_obs", "exposure_id", "obs_start",
    "tel_az_deg", "tel_alt_deg",
    "sun_az_deg", "sun_alt_deg",
    "moon_az_deg", "moon_alt_deg",
    "moon_sep_deg", "moon_illum_percent",
]

In [ ]:
alignment_summary.sort("sun_alt_deg", reverse=True)

In [ ]:
alignment_summary[
    "day_obs", "exposure_id", "obs_start",
    "tel_az_deg", "tel_alt_deg",
    "sun_az_deg", "sun_alt_deg",
    "moon_az_deg", "moon_alt_deg",
    "moon_sep_deg", "moon_illum_percent",
]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1) Convert table -> pandas DataFrame
alignment_df = alignment_summary.to_pandas()  # Astropy Table supports this

# 2) Parse day_obs (YYYYMMDD int/string) -> datetime
alignment_df["day_obs_dt"] = pd.to_datetime(alignment_df["day_obs"].astype(str), format="%Y%m%d")

# 3) Aggregate per night (median is a good default), then EWMA
alignment_nightly = (
    alignment_df.groupby("day_obs_dt")["sun_alt_deg"]
      .median()
      .sort_index()
)

# Option A: EWMA by "nights" (works fine if you have most nights)
alignment_ewma = nightly.ewm(span=5, adjust=False).mean()

# Option B (often better): time-aware EWMA, handles missing nights naturally
# ewma = nightly.ewm(halflife="7D", times=nightly.index).mean()

# 4) Plot
plt.figure(figsize=(10, 4))

plt.plot(
    alignment_ewma.index,
    alignment_ewma.values,
    linewidth=2.0,
    alpha=0.6,
    zorder=1,
    label="EWMA (span=10 nights)",
)

plt.scatter(
    alignment_nightly.index,
    alignment_nightly.values,
    s=25,
    alpha=0.7,
    zorder=3,
    edgecolors="0.2",
    label="Nightly median",
)

plt.axhline(-10, color="g", linestyle="--", linewidth=1, alpha=0.6)
plt.axhline(-12, color="r", linestyle="--", linewidth=1, alpha=0.6)

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.xlabel("Date)")
plt.ylabel("Solar Elevation (deg)")
plt.title("Starting Solar Elevation of Initial Alignment Observations")
plt.grid(True, alpha=0.3)
plt.ylim(-15,-8.0)

plt.tight_layout()
plt.show()

In [ ]:
alignment_exposures

***
Combined Timing
***

In [ ]:
# ------------------------------------------------------------
# 1. Alignment exposures
# ------------------------------------------------------------
align_df = alignment_exposures.to_pandas().copy()

align_df["obs_start"] = pd.to_datetime(
    align_df["obs_start"],
    format="mixed",
    utc=True,
)

align_df["obs_end"] = pd.to_datetime(
    align_df["obs_end"],
    format="mixed",
    utc=True,
)

align_df = align_df.sort_values(["day_obs", "obs_start"]).copy()

# ------------------------------------------------------------
# 2. Identify separate alignment sequences
#    New sequence if:
#      - seq_num jumps by > 2
#      - OR time gap between exposures is > 2 min
# ------------------------------------------------------------
align_df["prev_obs_end"] = (
    align_df.groupby("day_obs")["obs_end"]
    .shift()
)

align_df["seq_num_diff"] = (
    align_df.groupby("day_obs")["seq_num"]
    .diff()
)

align_df["time_gap_min"] = (
    align_df["obs_start"] - align_df["prev_obs_end"]
).dt.total_seconds() / 60.0

align_df["new_alignment_sequence"] = (
    (align_df["seq_num_diff"] > 1)
).fillna(False)

align_df["alignment_sequence"] = (
    align_df.groupby("day_obs")["new_alignment_sequence"]
    .cumsum()
)

alignment_sequences = (
    align_df
    .groupby(["day_obs", "alignment_sequence"])
    .agg(
        first_alignment_exposure=("exposure_id", "first"),
        last_alignment_exposure=("exposure_id", "last"),
        alignment_start=("obs_start", "min"),
        alignment_end=("obs_end", "max"),
        n_alignment_exposures=("exposure_id", "count"),
        first_alignment_seq_num=("seq_num", "first"),
        last_alignment_seq_num=("seq_num", "last"),
    )
    .reset_index()
)

alignment_sequences["alignment_duration_min"] = (
    alignment_sequences["alignment_end"]
    - alignment_sequences["alignment_start"]
).dt.total_seconds() / 60.0

# ------------------------------------------------------------
# 3. First survey exposure per night
# ------------------------------------------------------------
survey_df = summary.to_pandas().copy()

survey_df["obs_start"] = pd.to_datetime(
    survey_df["obs_start"],
    format="mixed",
    utc=True,
)

survey_start_timing = (
    survey_df
    .sort_values(["day_obs", "obs_start"])
    .groupby("day_obs")
    .agg(
        first_survey_start=("obs_start", "first"),
        first_survey_exposure=("exposure_id", "first"),
        first_survey_seq_num=("seq_num", "first"),
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. Match alignment sequences to first survey exposure
# ------------------------------------------------------------
candidate_sequences = alignment_sequences.merge(
    survey_start_timing,
    on="day_obs",
    how="inner",
)

# Only sequences ending before the first survey exposure starts
candidate_sequences = candidate_sequences[
    candidate_sequences["alignment_end"] <= candidate_sequences["first_survey_start"]
].copy()

# Require the alignment sequence to end immediately before survey starts in seq_num
candidate_sequences["seq_num_gap_to_survey"] = (
    candidate_sequences["first_survey_seq_num"]
    - candidate_sequences["last_alignment_seq_num"]
)

candidate_sequences["is_immediately_before_survey"] = (
    candidate_sequences["seq_num_gap_to_survey"] == 1
)

# Count how many alignment sequences occurred before survey start each night
n_pre_survey_sequences = (
    candidate_sequences
    .groupby("day_obs")["alignment_sequence"]
    .nunique()
    .rename("n_pre_survey_alignment_sequences")
    .reset_index()
)

candidate_sequences = candidate_sequences.merge(
    n_pre_survey_sequences,
    on="day_obs",
    how="left",
)

# Keep nights where the alignment sequence is immediately before the first survey image
nightly_timing = candidate_sequences[
    (candidate_sequences["is_immediately_before_survey"])
].copy()

# ------------------------------------------------------------
# 5. Timing breakdown
# ------------------------------------------------------------
nightly_timing["alignment_to_survey_min"] = (
    nightly_timing["first_survey_start"]
    - nightly_timing["alignment_end"]
).dt.total_seconds() / 60.0

nightly_timing["total_alignment_to_survey_start_min"] = (
    nightly_timing["first_survey_start"]
    - nightly_timing["alignment_start"]
).dt.total_seconds() / 60.0

nightly_timing = nightly_timing.sort_values("day_obs").reset_index(drop=True)


# ------------------------------------------------------------
# 6. Compact output
# ------------------------------------------------------------
nightly_timing[
    [
        "day_obs",
        "alignment_sequence",
        "n_pre_survey_alignment_sequences",
        "first_alignment_exposure",
        "last_alignment_exposure",
        "first_survey_exposure",
        "first_alignment_seq_num",
        "last_alignment_seq_num",
        "first_survey_seq_num",
        "seq_num_gap_to_survey",
        "n_alignment_exposures",
        "alignment_start",
        "alignment_end",
        "first_survey_start",
        "alignment_duration_min",
        "alignment_to_survey_min",
        "total_alignment_to_survey_start_min",
    ]
]

In [ ]:
nightly_timing.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plot_df = nightly_timing.copy()

# ------------------------------------------------------------
# Add evening -12 deg crossing
# ------------------------------------------------------------
plot_df["sun_minus12_time"] = [
    evening_sun_alt_crossing(day_obs, target_alt=-12.0)
    for day_obs in plot_df["day_obs"]
]

for col in ["alignment_start", "alignment_end", "first_survey_start"]:
    plot_df[col] = pd.to_datetime(
        plot_df[col],
        format="mixed",
        utc=True,
    )

plot_df["sun_minus12_time"] = pd.to_datetime(
    [t.iso for t in plot_df["sun_minus12_time"]],
    utc=True,
)

plot_df["sun_minus10_time"] = [
    evening_sun_alt_crossing(day_obs, target_alt=-10.0)
    for day_obs in plot_df["day_obs"]
]

plot_df["sun_minus10_time"] = pd.to_datetime(
    [t.iso for t in plot_df["sun_minus10_time"]],
    utc=True,
)

# ------------------------------------------------------------
# Times relative to -12 deg
# ------------------------------------------------------------
plot_df["alignment_start_min"] = (
    plot_df["alignment_start"] - plot_df["sun_minus12_time"]
).dt.total_seconds() / 60.0

plot_df["alignment_end_min"] = (
    plot_df["alignment_end"] - plot_df["sun_minus12_time"]
).dt.total_seconds() / 60.0

plot_df["survey_start_min"] = (
    plot_df["first_survey_start"] - plot_df["sun_minus12_time"]
).dt.total_seconds() / 60.0

plot_df["sun_minus10_min"] = (
    plot_df["sun_minus10_time"] - plot_df["sun_minus12_time"]
).dt.total_seconds() / 60.0

plot_df = plot_df.sort_values(
    "day_obs",
    ascending=False,
).reset_index(drop=True)

plot_df["minus10_to_alignment_min"] = (
    plot_df["alignment_start_min"] - plot_df["sun_minus10_min"]
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
y = np.arange(len(plot_df))

colors = {
    "pre_align": "#F4A6A6",
    "align": "#4E79A7",
    "gap": "#F28E2B",
    "survey": "#1B7F3A",
}

fig, ax = plt.subplots(
    figsize=(12, 0.42 * len(plot_df) + 2)
)

ax.barh(
    y,
    plot_df["minus10_to_alignment_min"],
    left=plot_df["sun_minus10_min"],
    height=0.55,
    color=colors["pre_align"],
    alpha=0.55,
    label="Sun -10° → alignment start",
)

# Alignment duration
ax.barh(
    y,
    plot_df["alignment_duration_min"],
    left=plot_df["alignment_start_min"],
    height=0.55,
    color=colors["align"],
    alpha=0.9,
    label="Initial alignment",
)

# Gap before survey
ax.barh(
    y,
    plot_df["alignment_to_survey_min"],
    left=plot_df["alignment_end_min"],
    height=0.55,
    color=colors["gap"],
    alpha=0.85,
    label="Alignment → survey gap",
)

# Alignment start marker
ax.scatter(
    plot_df["alignment_start_min"],
    y,
    marker="o",
    s=42,
    facecolors="white",
    edgecolors=colors["align"],
    linewidths=1.2,
    zorder=5,
)

# Alignment end marker
ax.scatter(
    plot_df["alignment_end_min"],
    y,
    marker="o",
    s=42,
    facecolors="white",
    edgecolors=colors["align"],
    linewidths=1.2,
    zorder=5,
)

# Survey start marker
ax.scatter(
    plot_df["survey_start_min"],
    y,
    marker="D",
    s=42,
    facecolors=colors["survey"],
    edgecolors="white",
    linewidths=0.8,
    zorder=6,
    label="Survey start",
)

# Twilight reference
ax.axvline(
    0,
    color='k',
    linestyle="--",
    linewidth=1.5,
    alpha=0.8,
)

# -10 Deg solar elevation
ax.scatter(
    plot_df["sun_minus10_min"],
    y,
    marker="D",
    s=28,
    facecolors="white",
    edgecolors="0.25",
    linewidths=1.2,
    zorder=6,
    label="Sun = -10°",
)

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------
ax.set_yticks(y)
ax.set_yticklabels(plot_df["day_obs"].astype(str))

ax.set_xlabel("Minutes since Sun elevation = -12° from rubin_nights")
ax.set_ylabel("Night")
ax.set_title("Nightly Alignment-to-Survey Timing Breakdown")

ax.set_xlim(-10, 40)

ax.grid(
    True,
    axis="x",
    alpha=0.25,
)

ax.grid(
    False,
    axis="y",
)

ax.legend(
    loc="upper right",
    frameon=True,
)

from matplotlib.ticker import MultipleLocator

# Major ticks every 10 min
ax.set_xticks(np.arange(-10, 61, 10))

# Minor ticks every 5 min
ax.xaxis.set_minor_locator(MultipleLocator(5))

# Limit x range
ax.set_xlim(-16, 60)

# Optional: show minor grid lines
ax.grid(True, which="major", axis="x", alpha=0.5, linestyle='dashed')
ax.grid(True, which="minor", axis="x", alpha=0.5, linestyle='dotted')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 3.5),
    sharey=True,
)

hist_specs = [
    (
        "minus10_to_alignment_min",
        "Sun -10° → Alignment Start",
        "#F4A6A6",
    ),
    (
        "alignment_duration_min",
        "Alignment Duration",
        "#4E79A7",
    ),
    (
        "alignment_to_survey_min",
        "Alignment → Survey Gap",
        "#F28E2B",
    ),
]

for ax, (col, title, color) in zip(axes, hist_specs):

    ax.hist(
        plot_df[col],
        bins="auto",
        color=color,
        edgecolor="black",
        alpha=0.8,
    )

    median = plot_df[col].median()

    ax.axvline(
        median,
        color="k",
        linestyle="--",
        linewidth=1.5,
        label=f"Median = {median:.1f} min",
    )

    ax.set_title(title)
    ax.set_xlabel("Minutes")
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[0].set_ylabel("N Nights")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Build per-exposure lookup tables
# ------------------------------------------------------------
align_lookup = alignment_exposures.to_pandas().copy()
survey_lookup = summary.to_pandas().copy()

align_lookup = align_lookup[
    [
        "exposure_id",
        "azimuth",
        "altitude",
        "obs_start",
    ]
].rename(
    columns={
        "azimuth": "last_align_az",
        "altitude": "last_align_el",
        "obs_start": "last_align_obs_start",
    }
)

survey_lookup = survey_lookup[
    [
        "exposure_id",
        "azimuth",
        "altitude",
        "obs_start",
    ]
].rename(
    columns={
        "azimuth": "survey_az",
        "altitude": "survey_el",
        "obs_start": "survey_obs_start",
    }
)


# ------------------------------------------------------------
# 2. Merge pointing info onto nightly_timing
# ------------------------------------------------------------
sep_df = nightly_timing.copy()

sep_df = sep_df.merge(
    align_lookup,
    left_on="last_alignment_exposure",
    right_on="exposure_id",
    how="left",
).drop(columns=["exposure_id"])

sep_df = sep_df.merge(
    survey_lookup,
    left_on="first_survey_exposure",
    right_on="exposure_id",
    how="left",
).drop(columns=["exposure_id"])


# ------------------------------------------------------------
# 3. Compute Az / El separation
# ------------------------------------------------------------
# Wrapped azimuth difference in degrees, in range [-180, +180]
sep_df["delta_az_deg"] = (
    (sep_df["survey_az"] - sep_df["last_align_az"] + 180) % 360
) - 180

sep_df["abs_delta_az_deg"] = sep_df["delta_az_deg"].abs()

sep_df["delta_el_deg"] = sep_df["survey_el"] - sep_df["last_align_el"]
sep_df["abs_delta_el_deg"] = sep_df["delta_el_deg"].abs()

# Approximate combined Az/El offset in degrees
sep_df["azel_sep_deg"] = np.sqrt(
    sep_df["abs_delta_az_deg"]**2 + sep_df["abs_delta_el_deg"]**2
)

# Optional: remove bad/missing rows
sep_df = sep_df.dropna(
    subset=[
        "alignment_to_survey_min",
        "abs_delta_az_deg",
        "abs_delta_el_deg",
        "azel_sep_deg",
    ]
).copy()


# ------------------------------------------------------------
# 4. Plot: does orange-bar duration correlate with pointing move?
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4),
    sharey=True,
)

plot_specs = [
    ("abs_delta_az_deg", r"$|\Delta$Az$|$ (deg)"),
    ("abs_delta_el_deg", r"$|\Delta$El$|$ (deg)"),
    ("azel_sep_deg", r"Combined Az/El separation (deg)"),
]

for ax, (xcol, xlabel) in zip(axes, plot_specs):

    ax.scatter(
        sep_df[xcol],
        sep_df["alignment_to_survey_min"],
        s=45,
        alpha=0.75,
        edgecolors="0.2",
    )

    ax.set_xlabel(xlabel)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Alignment → survey gap (min)")

plt.tight_layout()
plt.show()